In [1]:
import subprocess

from typing import Dict, Union
from bsbolt.Utils.UtilityFunctions import retrieve_iupac

In [20]:
class StreamSim:
    def __init__(self, paired_end=False, sim_command=None):
        self.paired_end = paired_end
        self.sim_command = sim_command
        self.contig_variants = {}
        self.variant_contig = None
        

    def __iter__(self):
        sim = subprocess.Popen(self.sim_command,
                               stdout=subprocess.PIPE,
                               universal_newlines=True)
        variant_output = False
        sim_output = iter(sim.stdout.readline, '')
        read_pair = {1: None, 2: None}
        paired_count = 0
        while True:
            try:
                formatted_line = next(sim_output).strip()
            except StopIteration:
                break
            else:
                if formatted_line == 'Contig Variant Start':
                    variant_output = True
                elif variant_output:
                    variant_output = self.collect_variant_info(formatted_line)
                    if not variant_output and self.contig_variants:
                        yield self.variant_contig, self.contig_variants
                        self.contig_variants = {}
                else:
                    read_info = self.process_read_name(formatted_line)
                    seq = next(sim_output).strip()
                    comment = next(sim_output).strip()
                    qual = self.modify_qual(next(sim_output).strip())
                    read_info.update(dict(comment=comment, seq=seq, qual=qual))
                    read_pair[read_info['pair']] = read_info
                    paired_count += 1
                    if paired_count == 2:
                        assert read_pair[1]['read_id'] == read_pair[2]['read_id']
                        yield False, read_pair
                        read_pair = {1: None, 2: None}
                        paired_count = 0

    @staticmethod
    def modify_qual(quality: str) -> str:
        # modify start position to ensure proper qual handling downstream
        quality_split = list(quality)
        qual_start = int(ord(quality_split[0]) - 33)
        quality_split[0] = chr(qual_start + 32)
        return ''.join(quality_split)

    def collect_variant_info(self, formatted_line):
        if formatted_line == 'Contig Variant End':
            return False
        variant_info = self.process_variant_line(formatted_line)
        assert variant_info['pos'] not in self.contig_variants
        self.contig_variants[variant_info['pos']] = variant_info
        self.variant_contig = variant_info['chrom']
        return True

    @staticmethod
    def process_variant_line(formatted_line):
        chrom, pos, reference, alt, heterozygous = formatted_line.split('\t')
        het = True if heterozygous == '+' else False
        pos = int(pos)
        indel = 0
        iupac = None
        if reference == '-':
            indel = 1
        elif alt == '-':
            indel = -1
        else:
            iupac = retrieve_iupac(alt)
        return dict(chrom=chrom, pos=pos, reference=reference, alt=alt, heterozygous=het,
                    indel=indel, iupac=iupac)

    @staticmethod
    def process_read_name(formatted_line: str) -> Dict[str, Union[str, int]]:
        read_info = formatted_line.split(':')
        chrom, start, end, insert_size, read_id, cigar, pair, c_base_info, g_base_info = read_info
        return dict(chrom=chrom.replace('@', ''), start=int(start), end=int(end),
                    insert_size=insert_size, read_id=read_id, cigar=cigar, pair=int(pair),
                    c_base_info=c_base_info, g_base_info=g_base_info)

In [21]:
paired_end = False
sim_command = ['/home/wbguo/iproject/BSBolt/bsbolt/External/WGSIM/wgsim',
 '-1',
 '100',
 '-2',
 '100',
 '-e',
 '0.005',
 '-d',
 '400',
 '-s',
 '25',
 '-r',
 '0.001',
 '-R',
 '0.15',
 '-X',
 '0.15',
 '-S',
 '-1',
 '-A',
 '0.05',
 '-I',
 '100',
 '-h',
 '0',
 '-N',
 '392320',
 '/home/wbguo/iproject/BSBolt/tests/TestData/BSB_test.fa']

In [22]:
" ".join(sim_command)

'/home/wbguo/iproject/BSBolt/bsbolt/External/WGSIM/wgsim -1 100 -2 100 -e 0.005 -d 400 -s 25 -r 0.001 -R 0.15 -X 0.15 -S -1 -A 0.05 -I 100 -h 0 -N 392320 /home/wbguo/iproject/BSBolt/tests/TestData/BSB_test.fa'

In [28]:
i=0
for variant_contig, sim_data in StreamSim(paired_end, sim_command):
    i +=1
    if i == 2:
        print(sim_data)
        break

{1: {'chrom': 'chr10', 'start': 283878, 'end': 283978, 'insert_size': '300', 'read_id': '0', 'cigar': 'MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM', 'pair': 1, 'c_base_info': '4_0,15_0,19_0,20_0,25_0,30_0,42_0,43_0,46_0,49_0,54_0,55_0,56_0,61_0,63_0,66_0,70_0,77_0,79_0,83_0,85_0,86_0,97_0,98_0,', 'g_base_info': '2_0,3_0,6_0,7_0,8_0,12_0,13_0,14_0,18_0,23_0,27_0,28_0,29_0,32_0,34_0,36_0,37_0,38_0,39_0,45_0,47_0,51_0,53_0,58_0,65_0,68_0,76_0,88_0,90_0,93_0,95_0,', 'comment': '+', 'seq': 'ATGGCTGGGTTTGGGCATGCCATGACAGGGCTGTGTGGGGTTCCAGCGACTGTGCCCTGATCACTGCAGTCATTAAGCTCATTCACCAGAGAAGTGACCT', 'qual': '7888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888888'}, 2: {'chrom': 'chr10', 'start': 284078, 'end': 284178, 'insert_size': '300', 'read_id': '0', 'cigar': 'MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM', 'pair': 2, 'c_base_info': '5_0,

[wgsim] seed = 1653433808
[wgsim_core] calculating the total length of the reference sequence...
[wgsim_core] 6 sequences, total length: 1961600


In [25]:
sim_data

{650: {'chrom': 'chr10',
  'pos': 650,
  'reference': 'A',
  'alt': 'W',
  'heterozygous': True,
  'indel': 0,
  'iupac': ('A', 'T')},
 735: {'chrom': 'chr10',
  'pos': 735,
  'reference': 'T',
  'alt': 'K',
  'heterozygous': True,
  'indel': 0,
  'iupac': ('G', 'T')},
 2503: {'chrom': 'chr10',
  'pos': 2503,
  'reference': 'A',
  'alt': 'C',
  'heterozygous': False,
  'indel': 0,
  'iupac': ('C',)},
 4438: {'chrom': 'chr10',
  'pos': 4438,
  'reference': 'A',
  'alt': 'W',
  'heterozygous': True,
  'indel': 0,
  'iupac': ('A', 'T')},
 5646: {'chrom': 'chr10',
  'pos': 5646,
  'reference': 'C',
  'alt': 'S',
  'heterozygous': True,
  'indel': 0,
  'iupac': ('G', 'C')},
 5921: {'chrom': 'chr10',
  'pos': 5921,
  'reference': 'G',
  'alt': 'C',
  'heterozygous': False,
  'indel': 0,
  'iupac': ('C',)},
 7029: {'chrom': 'chr10',
  'pos': 7029,
  'reference': 'C',
  'alt': 'S',
  'heterozygous': True,
  'indel': 0,
  'iupac': ('G', 'C')},
 7338: {'chrom': 'chr10',
  'pos': 7338,
  'referen

In [26]:
variant_contig

'chr10'